# Notebook 01: ทดลอง Web Scraping เว็บไซต์ Yuedpao.com

โน้ตบุ๊กนี้ใช้สำหรับการทดสอบดึงข้อมูลจากเว็บไซต์ **Yuedpao (https://www.yuedpao.com/)** โดยมี 2 วิธีหลัก:
1. **Requests + BeautifulSoup4 (bs4)**: ดึง HTML แบบ Static โดยตรง
2. **Playwright (Async inside Thread)**: จำลอง Browser สำหรับเว็บ Dynamic / Single Page App พร้อมระบบเลื่อน Scroll เมาส์ลงไปด้านล่างสุดเพื่อโหลดข้อมูลแบบ Lazy Loading (แก้ไขปัญหา `NotImplementedError` บน Windows Jupyter Notebook)

## 1. วิธีที่ 1: ใช้ Requests + BeautifulSoup (bs4)

In [8]:
import requests
from bs4 import BeautifulSoup

# กำหนด Target URL และ Headers เพื่อจำลอง Browser ทั่วไป
url = "https://www.yuedpao.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "th,en-US;q=0.9,en;q=0.8"
}

# ส่ง Request ไปยังเว็บปลายทาง
response = requests.get(url, headers=headers, timeout=15)
print(f"Status Code: {response.status_code}")

# ใช้ BeautifulSoup อ่านและแปลง HTML
soup = BeautifulSoup(response.text, "html.parser")

# ตรวจสอบ Title และข้อมูลพื้นฐาน
page_title = soup.title.string.strip() if soup.title else "No Title"
print(f"Page Title: {page_title}")
print(f"HTML Length: {len(response.text):,} characters")

# ตัวอย่าง: ค้นหาลิงก์และรูปภาพที่พบในหน้าแรก
links = soup.find_all("a", href=True)
images = soup.find_all("img")
print(f"Found {len(links)} links, {len(images)} images in static HTML")

Status Code: 200
Page Title: Yuedpao ยืดเปล่า ยืดแต่ไม่ย้วย ศูนย์รวมเสื้อยืดแบรนด์ไทย ผ้านุ่มใส่สบาย
HTML Length: 13,099 characters
Found 0 links, 0 images in static HTML


## 2. วิธีที่ 2: ใช้ Playwright พร้อม Auto-Scroll ลงด้านล่างสุด (เวอร์ชันสำหรับ Windows Jupyter)

> **แก้ปัญหา NotImplementedError บน Windows:**
> เนื่องจาก Jupyter Notebook บน Windows รันด้วย `SelectorEventLoop` ซึ่งไม่รองรับ subprocesses ของ Playwright 
> เราจึงจำเป็นต้องย้ายการทำงานของ Playwright ไปทำใน Thread ใหม่แยกต่างหาก และบังคับใช้ `ProactorEventLoop` ใน Thread นั้น

In [9]:
import sys
import asyncio
import threading
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

def scrape_yuedpao_with_scroll(url: str = "https://www.yuedpao.com/"):
    html_content = None
    exception = None

    def worker():
        nonlocal html_content, exception
        try: 
            # 1. สร้าง Event Loop ใหม่เฉพาะสำหรับ Thread นี้
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            # 2. บังคับใช้ ProactorEventLoop บน Windows เพื่อให้รองรับ subprocesses ของ Playwright
            if sys.platform == 'win32':
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
            
            # ฟังก์ชันภายในสำหรับรัน Playwright แบบ async
            async def run_playwright():
                async with async_playwright() as p:
                    # เปิด Browser แบบ Headless
                    browser = await p.chromium.launch(headless=True)
                    page = await browser.new_page()
                    
                    # ตั้งค่าขนาดหน้าต่างเว็บ
                    await page.set_viewport_size({"width": 1280, "height": 800})
                    
                    print(f"Navigating to: {url} ...")
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await page.wait_for_timeout(2000)
                    
                    # ทำการเลื่อน Scroll ลงไปด้านล่างสุดเรื่อยๆ เพื่อโหลดเนื้อหาแบบ Lazy Load
                    print("Scrolling down to the bottom of the page...")
                    last_height = await page.evaluate("document.body.scrollHeight")
                    
                    scroll_step = 600
                    max_scroll_attempts = 30
                    
                    for i in range(max_scroll_attempts):
                        # เลื่อนลง
                        await page.evaluate(f"window.scrollBy(0, {scroll_step})")
                        await page.wait_for_timeout(800)  # หน่วงเวลาให้ข้อมูลโหลด
                        
                        new_height = await page.evaluate("document.body.scrollHeight")
                        current_scroll_pos = await page.evaluate("window.pageYOffset + window.innerHeight")
                        
                        # ถ้าเลื่อนถึงจุดล่างสุดแล้ว
                        if current_scroll_pos >= new_height and new_height == last_height:
                            print(f"Reached bottom at attempt {i+1} (Scroll Height: {new_height}px)")
                            break
                        last_height = new_height
                    
                    # รอเพิ่มเติมให้มั่นใจว่าโหลดข้อมูลเสร็จสมบูรณ์
                    await page.wait_for_timeout(2000)
                    
                    # ดึง HTML ที่เรนเดอร์เสร็จแล้ว
                    content = await page.content()
                    await browser.close()
                    return content
            
            html_content = loop.run_until_complete(run_playwright())
        except Exception as e:
            exception = e
        finally:
            loop.close()

    # รันการทำงานของ Playwright ใน Thread แยก
    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    
    if exception:
        raise exception
    return html_content

# เรียกใช้ฟังก์ชันเพื่อดึง HTML
html_content = scrape_yuedpao_with_scroll("https://www.yuedpao.com/")

# นำ HTML ที่ได้มาวิเคราะห์ต่อด้วย BeautifulSoup
soup_dynamic = BeautifulSoup(html_content, "html.parser")
print(f"\nDynamic HTML Length: {len(html_content):,} characters")
print(f"Page Title: {soup_dynamic.title.string.strip() if soup_dynamic.title else 'No Title'}")

# ตรวจสอบจำนวน elements ที่เรนเดอร์หลัง Scroll
dynamic_images = soup_dynamic.find_all("img")
dynamic_links = soup_dynamic.find_all("a")
print(f"Found {len(dynamic_links)} links, {len(dynamic_images)} images after dynamic scroll")

Navigating to: https://www.yuedpao.com/ ...
Scrolling down to the bottom of the page...
Reached bottom at attempt 11 (Scroll Height: 6821px)

Dynamic HTML Length: 431,947 characters
Page Title: Yuedpao ยืดเปล่า ยืดแต่ไม่ย้วย ศูนย์รวมเสื้อยืดแบรนด์ไทย ผ้านุ่มใส่สบาย
Found 170 links, 137 images after dynamic scroll


## 3. วิธีที่ 3: ค้นหาแถบเมนูด้านข้าง (Hamburger Menu Drawer) และตรวจหา Class เฉพาะ

ในส่วนนี้เราจะทำการดึงหน้าเว็บด้วย viewport แบบ Mobile เพื่อให้ปุ่ม Hamburger Menu แสดงขึ้นมา จากนั้นจะจำลองการกดเปิด Drawer เพื่อไปค้นหาว่ามี class ที่มีชื่อว่า `MuiTypography-root MuiTypography-body pointer-cursor css-1dwwjt3` สำหรับรายการเมนู เช่น **ULTRA FLOW (กีฬา)** อยู่ทั้งหมดกี่อัน

In [10]:
import sys
import asyncio
import threading
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

def scrape_yuedpao_mobile_drawer(url: str = "https://www.yuedpao.com/"):
    html_content = None
    exception = None

    def worker():
        nonlocal html_content, exception
        try:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            if sys.platform == 'win32':
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
                
            async def run():
                async with async_playwright() as p:
                    browser = await p.chromium.launch(headless=True)
                    page = await browser.new_page()
                    
                    # จำลองเปิดเป็นหน้าจอมือถือเพื่อรองรับ Hamburger Menu
                    await page.set_viewport_size({"width": 375, "height": 812})
                    
                    print(f"Navigating to: {url} on mobile viewport...")
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await page.wait_for_timeout(3000)
                    
                    # หา SVG Hamburger Menu ที่มี Path ที่ระบุ
                    svgs = await page.query_selector_all("svg")
                    clicked = False
                    for svg in svgs:
                        # ค้นหา SVG ที่มีค่า path d ตรงกับปุ่มเปิด Menu ข้างของเว็บ
                        has_path = await svg.evaluate("""
                            (el) => {
                                const path = el.querySelector('path');
                                return path && path.getAttribute('d') === 'M13,16H3a1,1,0,0,0,0,2H13a1,1,0,0,0,0-2ZM3,8H21a1,1,0,0,0,0-2H3A1,1,0,0,0,3,8Zm18,3H3a1,1,0,0,0,0,2H21a1,1,0,0,0,0-2Z';
                            }
                        """)
                        if has_path:
                            print("Hamburger Menu button found! Clicking to open side navigation...")
                            await svg.click()
                            await page.wait_for_timeout(2500)  # รอ Drawer กางออกมาจนสุด
                            clicked = True
                            break
                    
                    if not clicked:
                        print("Warning: Hamburger Menu SVG path not found/clicked")
                    
                    content = await page.content()
                    await browser.close()
                    return content
            
            html_content = loop.run_until_complete(run())
        except Exception as e:
            exception = e
        finally:
            loop.close()

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    
    if exception:
        raise exception
    return html_content

In [14]:
# ดึงข้อมูล HTML จากเวอร์ชัน Mobile หลังเปิด Drawer
mobile_html = scrape_yuedpao_mobile_drawer("https://www.yuedpao.com/")
soup_mobile = BeautifulSoup(mobile_html, "html.parser")

# 1. ค้นหาทุก span ที่ใช้คลาสสิกของเมนูย่อย
# ในที่นี้คือ: class="MuiTypography-root MuiTypography-body pointer-cursor css-1dwwjt3"
target_class = ["MuiTypography-root", "MuiTypography-body", "pointer-cursor", "css-1dwwjt3"]

# ดึงเฉพาะ span ที่มีคลาสเหล่านี้ครบ
matching_spans = []
for span in soup_mobile.find_all("span"):
    classes = span.get("class", [])
    # ตรวจสอบว่าคลาสสำคัญมีครบถ้วน
    if all(cls in classes for cls in target_class):
        matching_spans.append(span)

print(f"\nTotal spans matching classes {target_class}: {len(matching_spans)}")
print("--------------------------------------------------")

for s in matching_spans:
    text = s.get_text(strip=True)
    # แสดงเมนูทั้งหมดที่ค้นพบ
    print(f"- Menu Item Found: {text}")

Navigating to: https://www.yuedpao.com/ on mobile viewport...
Hamburger Menu button found! Clicking to open side navigation...

Total spans matching classes ['MuiTypography-root', 'MuiTypography-body', 'pointer-cursor', 'css-1dwwjt3']: 34
--------------------------------------------------
- Menu Item Found: ULTRA FLOW (กีฬา)
- Menu Item Found: ULTRASOFT NON-IRON (เสื้อยืด)
- Menu Item Found: ULTIMATE COLLECTION (Work wear)
- Menu Item Found: SMOOTH SKIN
- Menu Item Found: SMOOTH FLEX JEANS (กางเกงยีนส์ยืด)
- Menu Item Found: SMOOTH STRETCH JEANS
- Menu Item Found: SOFT TECH UNWEAR
- Menu Item Found: UNWEAR
- Menu Item Found: RIB BRA
- Menu Item Found: RIB COOL MOOD
- Menu Item Found: KODNUM
- Menu Item Found: SIGNATURE
- Menu Item Found: OVERSIZED
- Menu Item Found: OVERSIZE TIMELESS
- Menu Item Found: FEATHER COMFORT
- Menu Item Found: YUEDPAO COLLECTION
- Menu Item Found: FLEECE AIR FLOW COLLECTION
- Menu Item Found: COLLAB COLLECTION
- Menu Item Found: ECOTECH
- Menu Item Found: YXZ